In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a program that applies a Gaussian blur filter to a 2D image. Given an input image represented as a floating-point array and a Gaussian kernel, the program should compute the convolution of the image with the kernel.
  All inputs and outputs are stored in row-major order.
</p>

<p>
  The Gaussian blur is performed by convolving each pixel with a weighted average of its neighbors, where the weights are determined by the Gaussian kernel. For each output pixel at position (i, j), the value is calculated as:

  $$ output[i, j] = \sum_{m=-k_h/2}^{k_h/2} \sum_{n=-k_w/2}^{k_w/2} input[i+m, j+n] \times kernel[m+k_h/2, n+k_w/2] $$

  where $k_h$ and $k_w$ are the kernel height and width.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>output</code> array</li>
  <li>Handle boundary conditions by using zero-padding (treat values outside the image boundary as zeros)</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  image (5, 5) = [
    [1.0, 2.0, 3.0, 4.0, 5.0],
    [6.0, 7.0, 8.0, 9.0, 10.0],
    [11.0, 12.0, 13.0, 14.0, 15.0],
    [16.0, 17.0, 18.0, 19.0, 20.0],
    [21.0, 22.0, 23.0, 24.0, 25.0]
  ]

  kernel (3, 3) = [
    [0.0625, 0.125, 0.0625],
    [0.125, 0.25, 0.125],
    [0.0625, 0.125, 0.0625]
  ]

Output:
  output (5, 5) = [
    [1.6875, 2.75, 3.5, 4.25, 3.5625],
    [4.75, 7.0, 8.0, 9.0, 7.25],
    [8.5, 12.0, 13.0, 14.0, 11.0],
    [12.25, 17.0, 18.0, 19.0, 14.75],
    [11.0625, 15.25, 16.0, 16.75, 12.9375]
  ]

</pre>

<h2>Example 2:</h2>
<pre>
Input:
  image (3, 3) = [
    [10.0, 20.0, 30.0],
    [40.0, 50.0, 60.0],
    [70.0, 80.0, 90.0]
  ]

  kernel (3, 3) = [
    [0.1, 0.1, 0.1],
    [0.1, 0.2, 0.1],
    [0.1, 0.1, 0.1]
  ]

Output:
  output (3, 3) = [
    [13.0, 23.0, 19.0],
    [31.0, 50.0, 39.0],
    [31.0, 47.0, 37.0]
  ]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>input_rows</code>, <code>input_cols</code> ≤ 4096</li>
  <li>3 ≤ <code>kernel_rows</code>, <code>kernel_cols</code> ≤ 21</li>
  <li>Both <code>kernel_rows</code> and <code>kernel_cols</code> will be odd numbers</li>
  <li>All kernel values will be non-negative and sum to 1.0 (normalized)</li>

  <li>Performance is measured with <code>input_cols</code> = 512, <code>input_rows</code> = 512, <code>kernel_cols</code> = 7, <code>kernel_rows</code> = 7</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, kernel, output are device pointers
extern "C" void solve(const float* input, const float* kernel, float* output, int input_rows,
                      int input_cols, int kernel_rows, int kernel_cols) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, kernel, output are tensors on the GPU
@cute.jit
def solve(
    input: cute.Tensor,
    kernel: cute.Tensor,
    output: cute.Tensor,
    input_rows: cute.Int32,
    input_cols: cute.Int32,
    kernel_rows: cute.Int32,
    kernel_cols: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input, kernel are tensors on the GPU
@jax.jit
def solve(
    input: jax.Array,
    kernel: jax.Array,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    kernel: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    input_rows: Int32,
    input_cols: Int32,
    kernel_rows: Int32,
    kernel_cols: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, kernel, output are tensors on the GPU
def solve(
    input: torch.Tensor,
    kernel: torch.Tensor,
    output: torch.Tensor,
    input_rows: int,
    input_cols: int,
    kernel_rows: int,
    kernel_cols: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Gaussian Blur", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        input: torch.Tensor,
        kernel: torch.Tensor,
        output: torch.Tensor,
        input_rows: int,
        input_cols: int,
        kernel_rows: int,
        kernel_cols: int,
    ):
        input_2d = input.view(1, 1, input_rows, input_cols)
        kernel_2d = kernel.view(1, 1, kernel_rows, kernel_cols)
        pad_h = kernel_rows // 2
        pad_w = kernel_cols // 2
        result = torch.nn.functional.conv2d(input_2d, kernel_2d, padding=(pad_h, pad_w))
        output[:] = result.view(-1)

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "kernel": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "input_rows": (ctypes.c_int, "in"),
            "input_cols": (ctypes.c_int, "in"),
            "kernel_rows": (ctypes.c_int, "in"),
            "kernel_cols": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        input_rows, input_cols = 5, 5
        kernel_rows, kernel_cols = 3, 3
        input = torch.tensor(
            [
                1.0,
                2.0,
                3.0,
                4.0,
                5.0,
                6.0,
                7.0,
                8.0,
                9.0,
                10.0,
                11.0,
                12.0,
                13.0,
                14.0,
                15.0,
                16.0,
                17.0,
                18.0,
                19.0,
                20.0,
                21.0,
                22.0,
                23.0,
                24.0,
                25.0,
            ],
            device="cuda",
            dtype=dtype,
        )
        kernel = torch.tensor(
            [0.0625, 0.125, 0.0625, 0.125, 0.25, 0.125, 0.0625, 0.125, 0.0625],
            device="cuda",
            dtype=dtype,
        )
        output = torch.empty(input_rows * input_cols, device="cuda", dtype=dtype)
        return {
            "input": input,
            "kernel": kernel,
            "output": output,
            "input_rows": input_rows,
            "input_cols": input_cols,
            "kernel_rows": kernel_rows,
            "kernel_cols": kernel_cols,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        device = "cuda"
        tests = []

        # basic_example
        tests.append(
            {
                "input": torch.tensor(
                    [
                        [1.0, 2.0, 3.0, 4.0, 5.0],
                        [6.0, 7.0, 8.0, 9.0, 10.0],
                        [11.0, 12.0, 13.0, 14.0, 15.0],
                        [16.0, 17.0, 18.0, 19.0, 20.0],
                        [21.0, 22.0, 23.0, 24.0, 25.0],
                    ],
                    device=device,
                    dtype=dtype,
                ).flatten(),
                "kernel": torch.tensor(
                    [[0.0625, 0.125, 0.0625], [0.125, 0.25, 0.125], [0.0625, 0.125, 0.0625]],
                    device=device,
                    dtype=dtype,
                ).flatten(),
                "output": torch.zeros((5, 5), device=device, dtype=dtype).flatten(),
                "input_rows": 5,
                "input_cols": 5,
                "kernel_rows": 3,
                "kernel_cols": 3,
            }
        )

        # identity_kernel
        tests.append(
            {
                "input": torch.tensor(
                    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]], device=device, dtype=dtype
                ).flatten(),
                "kernel": torch.tensor(
                    [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0]], device=device, dtype=dtype
                ).flatten(),
                "output": torch.zeros((3, 3), device=device, dtype=dtype).flatten(),
                "input_rows": 3,
                "input_cols": 3,
                "kernel_rows": 3,
                "kernel_cols": 3,
            }
        )

        # all_ones_input
        tests.append(
            {
                "input": torch.ones((4, 4), device=device, dtype=dtype).flatten(),
                "kernel": torch.full((3, 3), 0.111111, device=device, dtype=dtype).flatten(),
                "output": torch.zeros((4, 4), device=device, dtype=dtype).flatten(),
                "input_rows": 4,
                "input_cols": 4,
                "kernel_rows": 3,
                "kernel_cols": 3,
            }
        )

        # single_pixel
        tests.append(
            {
                "input": torch.tensor([[42.0]], device=device, dtype=dtype).flatten(),
                "kernel": torch.tensor([[1.0]], device=device, dtype=dtype).flatten(),
                "output": torch.zeros((1, 1), device=device, dtype=dtype).flatten(),
                "input_rows": 1,
                "input_cols": 1,
                "kernel_rows": 1,
                "kernel_cols": 1,
            }
        )

        # large_random
        input_large = torch.empty((32, 32), device=device, dtype=dtype).uniform_(-10.0, 10.0)
        kernel_large = torch.empty((5, 5), device=device, dtype=dtype).uniform_(0.0, 1.0)
        tests.append(
            {
                "input": input_large.flatten(),
                "kernel": kernel_large.flatten(),
                "output": torch.zeros((32, 32), device=device, dtype=dtype).flatten(),
                "input_rows": 32,
                "input_cols": 32,
                "kernel_rows": 5,
                "kernel_cols": 5,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        input_rows, input_cols = 512, 512
        kernel_rows, kernel_cols = 7, 7
        input = torch.empty(input_rows * input_cols, device="cuda", dtype=dtype).uniform_(
            0.0, 255.0
        )
        kernel = torch.empty(kernel_rows * kernel_cols, device="cuda", dtype=dtype).uniform_(
            0.0001, 0.02
        )
        output = torch.empty(input_rows * input_cols, device="cuda", dtype=dtype)
        return {
            "input": input,
            "kernel": kernel,
            "output": output,
            "input_rows": input_rows,
            "input_cols": input_cols,
            "kernel_rows": kernel_rows,
            "kernel_cols": kernel_cols,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
